## Main analysis

### Setup

In [ ]:
# parameters/cut-offs used
min_umi = 500
max_umi = 150000
min_genes = 250
max_mito = 1
max_doublet = 0.1

knn_n_neighbors = 20
knn_n_pcs = 30

# Auxiliary files: 
marker_list = "references/bridge_celltype_markers.csv"
gene_name_mapping = "references/t2g.txt"

In [ ]:
import snapatac2 as snap
import sys
import decoupler as dc
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import gc
sys.path.append('./utils')
from importing import download_h5ads, add_metadata_to_adata, build_adata, add_gene_names_to_adata, run_scrublet
from plotting import plot_knee_curve, plot_expression_heatmap, stacked_barplot_proportions
import seaborn as sns
import importlib
importlib.reload(sys.modules['importing'])
importlib.reload(sys.modules['plotting'])

### Joint Plot

In [ ]:
def label_pair(row):
    if row['RNA_QC'] == 1 and row['ATAC_QC'] == 1:
        return 'both'
    elif row['RNA_QC'] == 1:
        return 'RNA_only'
    elif row['ATAC_QC'] == 1:
        return 'ATAC_only'
    else:
        return 'neither'

In [ ]:
## RNA unfiltered obs (kept cells UMI >=10). Input file is afer running scRNA-preprocess.ipynb

rna_obs = pd.read_csv("results/unfiltered_rna_obs.tsv", sep = "\t")
rna_obs["RNA_QC"] = 0
rna_obs.loc[(rna_obs["total_counts"] >= 500) & 
          (rna_obs["n_genes_by_counts"] >= 250) &
          (rna_obs['pct_counts_mt'] <= max_mito), "RNA_QC"]=1

rna_obs

In [ ]:
## ATAC unfiltered obs. Input file is afer running scATAC-preprocess.ipynb
## WARNING: Really large file, not even filtered for fragments >=10 

atac_obs = pd.read_csv("results/unfilt-atac-obs.tsv", sep = "\t")

In [ ]:
atac_obs = atac_obs[atac_obs["n_fragments"] >=10 ]
atac_obs["ATAC_QC"] = 0
atac_obs.loc[
          (atac_obs["n_fragments"] >= 500) & 
          (atac_obs["tsse"] >= 5), "ATAC_QC"] = 1

atac_obs

In [ ]:
doublet_df = pd.read_csv("results/atac_data_obs.tsv", sep = "\t")
doublet_bc = [s.split(":")[1] for s in doublet_df.loc[doublet_df["doublet_probability"] > 0.7,"barcode"]] 

In [ ]:
print("ATAC barcodes with >=500 fragments: ", len(atac_obs.loc[(atac_obs["n_fragments"] >= 500),"barcode"]))
print("RNA barcodes with >=500 UMIs: ", len(rna_obs.loc[(rna_obs["total_counts"] >= 500),"cell_barcode"]))
print("# of doublets called: ", len(doublet_bc))

In [ ]:
joint = pd.merge(atac_obs, rna_obs, how = "outer", left_on= "barcode", right_on= "cell_barcode")
joint['RNA_QC'] = joint['RNA_QC'].fillna(0)
joint['ATAC_QC'] = joint['ATAC_QC'].fillna(0)

In [ ]:
joint["QC_pass"] = joint.apply(label_pair, axis=1)
joint.loc[joint["barcode"].isin(doublet_bc), "QC_pass"] = "Doublet"
joint["QC_pass"].value_counts()

In [ ]:
import seaborn as sns
plt.figure(figsize=(10, 10))

ax = sns.scatterplot(data = joint, x = "n_fragments", y = "total_counts", hue = "QC_pass", 
                     hue_order=["neither", "Doublet", "both", "ATAC_only", "RNA_only"],
                     palette=['darkgrey', 'tan', 'mediumpurple', 'chocolate','deepskyblue'], s=5, legend=False)

ax.set_xscale('log')
ax.set_yscale('log')
plt.show()

In [ ]:
gc.collect()

### scRNA processing

In [ ]:
adata = sc.read_h5ad("results/filtered-RNA.h5ad")

In [ ]:
filt_atac_obs = pd.read_csv("results/filt-atac-obs.tsv", sep = "\t")
filt_atac_obs["barcode"] = [s.split(":")[1] for s in atac_adata.obs_names]
filt_atac_obs["is_doublet"] = "False"
filt_atac_obs.loc[filt_atac_obs["doublet_probability"] >=0.7 ,"is_doublet"] = "True"

filt_rna_obs = adata.obs
filt_rna_obs_indexed = filt_rna_obs.set_index("cell_barcode")

filt_atac_obs_indexed = filt_atac_obs.set_index("barcode")
common_barcodes = list(set(filt_rna_obs_indexed.index) & set(filt_atac_obs_indexed.index))

filt_rna_obs_indexed["is_doublet"] = "Undetermined"
filt_rna_obs_indexed.loc[ common_barcodes, "is_doublet"] = filt_atac_obs_indexed.loc[common_barcodes, "is_doublet"]

In [ ]:
#remove doublets
adata = adata[adata.obs_names.isin( filt_rna_obs_indexed[filt_rna_obs_indexed["is_doublet"] != "True"].index)].copy()

sc.pp.normalize_total(adata, target_sum=1e4, layers=None, inplace=True) # Counts per 10k
sc.pp.log1p(adata, layer=None)
gc.collect()
# highly variable genes are used to compute the clustering 
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)

adatas = adata[:, adata.var.highly_variable]
gc.collect()
sc.pp.regress_out(adatas, ['pct_counts_mt','n_genes_by_counts'])
gc.collect()
sc.tl.pca(adatas, svd_solver='arpack')
print(sc.pl.pca_overview(adatas, color=["leiden", "sample"])) 
gc.collect()

In [ ]:
sc.pl.pca_variance_ratio(adatas, n_pcs=30, save="PCA_variance_ratio.svg")

In [ ]:
sc.pp.neighbors(adatas, n_neighbors=15, n_pcs=10) # put non standard settings near the top
gc.collect()
print("Clustering....")
sc.tl.leiden(adatas, resolution = 0.5)
sc.tl.umap(adatas, min_dist=0.2, spread=0.2)

adata.uns['neighbors'] = adatas.uns['neighbors']
adata.uns['leiden'] = adatas.uns['leiden']
adata.uns['umap'] = adatas.uns['umap']
adata.obs['leiden'] = adatas.obs['leiden']
adata.obsm = adatas.obsm
adata.obsp = adatas.obsp
gc.collect()

print(sc.pl.umap(adata, 
           color=['leiden'], 
           size=10, 
           legend_loc = 'on data', 
           legend_fontsize=15,
           show = False
          ))

### scATAC processing

In [ ]:
atac_adata = snap.read("results/filtered-ATAC.h5ad")

In [ ]:
filt_atac_obs_indexed = filt_atac_obs.set_index("barcode")
filt_atac_obs_indexed["cell_barcode"] = filt_atac_obs_indexed["sample"] + ":" + filt_atac_obs_indexed.index
#filt_atac_obs_indexed

selected_barcodes = filt_atac_obs_indexed.loc[
    filt_atac_obs_indexed["doublet_probability"] <= 0.7, "cell_barcode"
].tolist()

#### WARNING: this will modify the ATAC h5ad on disk!! Advise making a copy of this file before continuing this step.


In [ ]:
atac_adata.subset(obs_indices=selected_barcodes)

In [ ]:
snap.tl.spectral(atac_adata)
snap.tl.umap(atac_adata)
snap.pp.knn(atac_adata)
snap.tl.leiden(atac_adata)

In [ ]:
snap.pl.umap(atac_adata, color='leiden', interactive=False, out_file = "figures/atac_umap.svg")


### cell annotation

In [ ]:
import decoupler as dc

marker_df = pd.read_csv(marker_list)
marker_genes_dict = marker_df.groupby('celltype')['marker_gene'].apply(list).to_dict()
# Keep only the genes that are in the adata object
marker_genes_dict = {key: [gene for gene in value if gene in adata.var_names] for key, value in marker_genes_dict.items()}
#marker_genes_dict.pop("Lactosomatotrope")
valid_marker_genes_dict = {
    cell_type: [gene for gene in genes if gene in adata.var_names]
    for cell_type, genes in marker_genes_dict.items()
}
valid_marker_genes_dict

# Convert to decoupler-compatible format
brain_region_df = pd.DataFrame(
    [(region, gene) for region, genes in valid_marker_genes_dict.items() for gene in genes],
    columns=["source", "target"]
)

dc.run_ora(
    mat=adata,
    net=brain_region_df,
    source='source',
    target='target',
    use_raw=False
)

In [ ]:
acts = dc.get_acts(adata, obsm_key='ora_estimate')

# We need to remove inf and set them to the maximum value observed for pvals=0
acts_v = acts.X.ravel()
max_e = np.nanmax(acts_v[np.isfinite(acts_v)])
acts.X[~np.isfinite(acts.X)] = max_e

df = dc.rank_sources_groups(acts, groupby='leiden', reference='rest', method='t-test_overestim_var')
n_ctypes = 3
ctypes_dict = df.groupby('group').head(n_ctypes).groupby('group')['names'].apply(lambda x: list(x)).to_dict()

annotation_dict = df.groupby('group').head(1).set_index('group')['names'].to_dict()

adata.obs['celltype'] = [annotation_dict[clust] for clust in adata.obs['leiden']]

# Visualize
sc.pl.umap(adata, color='celltype')

### save h5ad outputs

In [ ]:
atac_adata.write("results/SHAREv2-ATAC-final.h5ad")

In [ ]:
adata.write_h5ad("results/SHAREv2-RNA-final.h5ad")

### DORCs

In [ ]:
%load_ext autoreload
%autoreload 2
import scprinter as scp
import pandas as pd
import os
import numpy as np
import scanpy as sc
import anndata
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

In [ ]:
adata_rna = anndata.read_h5ad(os.path.join('results/SHAREv2-RNA-final.h5ad'))
adata_atac = anndata.read_h5ad(os.path.join('results/SHAREv2-ATAC-final.h5ad'))

In [ ]:
adata_atac.obs.index = [xx.split(":")[1] for xx in adata_atac.obs.index]

In [ ]:
shared = list(set(adata_rna.obs.index) & set(adata_atac.obs.index))
len(shared)

In [ ]:
adata_atac = adata_atac[shared].copy()
adata_rna = adata_rna[shared].copy()

In [ ]:
tss_df = pd.read_csv('references/FigR_mm39TSSRanges.bed', sep='\t')
tss_df = tss_df.drop_duplicates('gene_name')
tss_df.index = tss_df['gene_name']

In [ ]:
import time
start = time.time()
dorc_all = scp.dorc.fast_gene_peak_corr(adata_atac,
                        adata_rna,
                        genome=scp.genome.mm39,
                        tss_df=tss_df,
                        gene_list=None,
                        window_pad_size=50000,
                        n_jobs=1,
                        n_bg=100,
                        pval_cut=None,
                        pos_only=True,
                        multimapping=False)
print ("takes", time.time() -start, "s")

In [ ]:
dorc_all.to_csv('results/dorc_all.tsv.gz',sep='\t')

In [ ]:
gene_list = scp.dorc.dorc_j_plot(dorc, cutoff=10, 
                                 label_top=10,
                                 return_gene_list=True, 
                )